# Implement local interpretable explanations (LIME)

**MedLIME Learning Track — Notebook 04/12** · Curriculum weeks 17–20 · Suggested time: 60–100 minutes

**Before starting:** Notebook 03; binary patch masks and linear regression.

**Learning goal:** Implement perturbation-based explanation without querying neural-network gradients.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## Question: which patch drives the output?
LIME samples masks, asks a black-box predictor for the score of each perturbed image, and fits a local weighted linear surrogate. The coefficients approximate local patch influence, not causal biological effects.

For masks \(z_i\), scores \(f(x(z_i))\), and proximity weights \(\pi_i\), estimate \(w,b\) by minimizing \(\sum_i\pi_i(f(x(z_i)) - b - z_i^\top w)^2\).

In [ ]:
x, lesion=synthetic_image(seed=6)
g=ToyBlackBox()  # Fixed detector; explanation only uses predict_proba.
print('Positive class score of original image:',round(float(g.predict_proba(x)[0]),3))
keep=random_patch_masks(n=180,seed=11)
perturbed=masked_examples(x,keep,fill='zero')
response=g.predict_proba(perturbed)
print('Shapes: masks, images, scores', keep.shape, perturbed.shape, response.shape)
show_images([x,perturbed[5],perturbed[10]],['original','sample perturbation','another perturbation'])
plt.show()

## Weighted regression
Masks close to the original image receive a higher weight. Interpret positive regression coefficients as patches whose presence tends to raise the local score under the chosen perturbation rule.

In [ ]:
weights=lime_kernel(keep,bandwidth=.75)
fit=fit_surrogate(keep,response,sample_weight=weights)
heatmap=coef_map(fit['coef'])
print('Local surrogate weighted R²:',round(fit['r2'],3))
print('Pixelwise AUPRC (positive class vs lesion mask):',round(auprc(lesion,heatmap),3))
fig=show_images([x,lesion.astype(float),heatmap],['synthetic image','true lesion mask','LIME patch coefficients'],cmap='viridis')
plt.show()

## Why not rely only on R²?
A surrogate can reproduce a model that relies on a scanner artifact instead of a lesion. Good local fitting and faithful medical localization measure different properties.

In [ ]:
plt.figure(figsize=(5,3))
plt.scatter(keep.mean(axis=1), response, c=weights, cmap='viridis',s=16)
plt.xlabel('Fraction of visible patches');plt.ylabel('Black-box score')
plt.title('Perturbation neighborhood');plt.colorbar(label='LIME proximity weight');plt.show()

## What you should now understand
- LIME is black-box: it needs model outputs but not weights or gradients.
- A local linear coefficient measures sensitivity under a chosen perturbation design.
- Good surrogate fit is not a guarantee of clinical localization.

## Try it yourself (do not just read the code)
1. Try `n=40`, `n=180`, and `n=500`; record saliency stability.
2. Change `bandwidth` to 0.3 and 1.2. What neighborhood does the regression emphasize?

## Exit ticket
**Question:** Why does changing the masking operator change the meaning of the LIME explanation?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.